<a href="https://colab.research.google.com/github/AlexF1789/homeworkTes/blob/homework1/homework1/homework1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Esercitazione software 1
## Analisi in frequenza di brani musicali


Autori:

- Fissolo Emanuele - *s323585*
- Flora Alessandro - *s321504*
- Giraudo Giacomo - *s321784*
- Intagliata Francesco - *s325961*

### Introduzione
L’obiettivo dell’homework è quello di analizzare delle tracce audio mediante le tecniche viste a lezione per l’elaborazione numerica dei segnali; in particolare la trasformazione del dominio del tempo nel dominio della frequenza tramite l’applicazione degli algoritmi di FFT e di DFT.

### 1) Operazioni preliminari
In questa cella vengono eseguite le operazioni preliminari **necessarie** al funzionamento del codice, quali la lettura della traccia audio, la definizione della dimensione della finestra da analizzare etc...

In [ ]:
# --- ESEGUIRE SOLO SU COLAB ---
%cd /content
!pip install numpy matplotlib soundfile
!git clone https://github.com/AlexF1789/homeworkTes.git
%cd homeworkTes/homework1
# ------------------------------

In [ ]:
#in questo blocco eseguiamo tutte le operazione preliminari necessarie per il funzionamento dei
#blocchi di codice successivi

import numpy as np
from cmath import exp, pi
import matplotlib.pyplot as plt
import tes, soundfile, os
from IPython.display import Audio, display

file_audio = str(os.path.join('input', 'CornfieldChase.oga'))
#file_audio = str(os.path.join('input', 'QueenBohemianRhapsody.oga'))

audio, fs = soundfile.read(file_audio)
audio = np.array(audio[:, 0], dtype=np.float32)

len_finestra = 0.5
N = int(fs*len_finestra)   #finestra da 0.5 secondi

Df = 1/(N*1/fs)
x_axis = [x*Df for x in range(int(-N/2),int(N/2))]

print(f"Estratti {len(audio)} campioni - campionati a {fs/1000:.1f}kHz - durata {len(audio)/fs:.2f} secondi")
print(f"Lavoriamo con finestre da {len_finestra:.1f} secondi - {N} campioni per finestra - risoluzione in frequenza {Df:.2f}Hz")

### Anteprima del file audio

La seguente cella consente di ottenere un'anteprima del file audio riproducendolo - la sua mancata riproduzione non inficia sul resto dell'elaborazione.

In [ ]:
display(Audio(file_audio))

### 2) Implementazioni della DFT

#### Introduzione
Come prima parte del lavoro andremo a effettuare delle prove sulla colonna sonora di Interstellar per paragonare l’efficienza degli algoritmi di FFT e DFT.

#### FFT da libreria Numpy
Iniziamo a calcolare la dft della prima finestra temporale della traccia audio utilizzando l'implementazione della FFT (Fast Fourier Transform) implementata nella libreria Numpy di Python

In [ ]:
# calcoliamo la FFT sugli N campioni della prima finestra della traccia audio
fft_py = np.abs(np.fft.fft(audio[0:N]))

# calcoliamo il limite di frequenza per avere la banda al 99% del segnale
max_freq_banda = tes.get_limite_banda(tes.get_spettro(fft_py), Df, 99)
print(f"La frequenza corrispondente alla banda al 99% è {max_freq_banda}Hz")

# plottiamo la trasformata
plt.plot(x_axis, np.fft.fftshift(fft_py))
plt.grid(True)
plt.axis([-max_freq_banda*1.25, max_freq_banda*1.25, 0, 1.25 * np.max(fft_py)])
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")
plt.show()

#### DFT implementata "a mano" in Python
Procediamo ora a implementare l'algoritmo di DFT a partire dalla definzione stessa della Discrete Fourier Transform.
La logica algoritmica è quella vista a lezione per cui si procede a sommare per ogni frequenza le rispettive componenti lungo tutta la finestra temporale del segnale passata in ingresso.

Sempre in Python è stata anche implementata una funzione di **shift** della DFT (di cui è possibile trovarne l'implementazione nel modulo **tes**), utile a ricomporre il risultato della trasformata per ridarle il significato reale.

Nell'esempio di implementazione applicheremo l'algoritmo ad una sotto-finestra temporale di dimensione estremamente ridotta, in quanto il calcolo su una finestra da 0.5 sec richiederebbe svariate decine di minuti

In [ ]:
#Implementazione della DFT (su una finestra di dimensioni ridotte)

# --- ATTENZIONE ---
# usare la finestra completa potrebbe portare a tempo di esecuzione
# superiori a 30 min, usare con prudenza

#segnale = audio[0:N]

segnale = audio[0:300]
Nt = len(segnale)
trasformata = [0 for _ in range(Nt)]
for k in range(0, Nt):
    somma = complex(0)

    for n in range(0, Nt):
        somma += segnale[n] * exp(-1j * 2 * pi * n * k/Nt)

    trasformata[k] = somma

dft_py = np.abs(np.array(trasformata))

#calcoliamo la Df e l'asse x per questa finestra temporale troncata
Df_t = 1/(Nt*1/fs)
x_axis_t = [x*Df_t for x in range(int(-Nt/2),int(Nt/2))]

# calcoliamo il limite di frequenza per avere la banda al 99% del segnale
max_freq_banda_t = tes.get_limite_banda(tes.get_spettro(dft_py), Df_t, 99)
print(f"La frequenza corrispondente alla banda al 99% è {max_freq_banda_t}Hz")

# plottiamo lo spettro del segnale
plt.plot(x_axis_t, tes.shift(dft_py))
plt.grid(True)
plt.axis([-max_freq_banda_t*1.25, max_freq_banda_t*1.25, 0, 1.25 * np.max(dft_py)])
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")
plt.show()

#### DFT implementata "a mano" in C
Spinti dall'inefficienza della soluzione precedente ci siamo chiesti quanto si possa imputare a Python e quanto invece, come esposto a lezione, fosse da imputare all'inefficienza dell'algoritmo.

Abbiamo pertanto, usando **ctypes**, implementato un wrapper in Python che richiama una funzione scritta in C che svolgesse le stesse operazioni della versione in Python.

In [ ]:
# calcoliamo la DFT sugli N campioni della prima finestra della traccia audio
dft_c = tes.dft_c(audio[0:N]) # qui non chiamiamo abs perché viene già calcolato da C

# calcoliamo il limite di frequenza per avere la banda al 99% del segnale
max_freq_banda = tes.get_limite_banda(tes.get_spettro(dft_c), Df, 99)
print(f"La frequenza corrispondente alla banda al 99% è {max_freq_banda}Hz")

# plottiamo lo spettro del segnale
plt.plot(x_axis, tes.shift(dft_c))
plt.grid(True)
plt.axis([-max_freq_banda*1.25, max_freq_banda*1.25, 0, 1.25 * np.max(dft_c)])
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")
plt.show()

### Versione con calcolo in parallelo

Su architetture **multicore** (come tutti i moderni computer) è possibile rendere il **calcolo parallelo** andando a sommare i vari contributi della DFT contemporaneamente.

Abbiamo dunque realizzato una versione di DFT in C che sfruttasse un approccio parallelo per il calcolo. In particolare il ciclo for più esterno non lavora più su un valore della frequenza k per volta ma su un numero legato alla potenza del **calcolatore utilizzato**.

In [ ]:
# calcoliamo la DFT sugli N campioni della prima finestra della traccia audio
dft_c = tes.dft_c_parallela(audio[0:N]) # qui non chiamiamo abs perché viene già calcolato da C

# calcoliamo il limite di frequenza per avere la banda al 99% del segnale
max_freq_banda = tes.get_limite_banda(tes.get_spettro(dft_c), Df, 99)
print(f"La frequenza corrispondente alla banda al 99% è {max_freq_banda}Hz")

# plottiamo lo spettro del segnale
plt.plot(x_axis, tes.shift(dft_c))
plt.grid(True)
plt.axis([-max_freq_banda*1.25, max_freq_banda*1.25, 0, 1.25 * np.max(dft_c)])
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")
plt.show()

#### Versione con calcolo su GPU
Avendo notato un discreto miglioramento con la parallelizzazione su più core dei calcoli della DFT abbiamo deciso di portare al limite questa soluzione. Per farlo ci avvaleremo delle migliaia di CUDA Cores presenti in una GPU per parallelizzare il calcolo dei termini della DFT, arrivando quindi a poter eseguire il calcolo di migliaia di campioni della DFT contemporaneamente

In [ ]:
# eseguire questa cella almeno 2 volte per raggiungere le massime prestazioni
# calcoliamo la DFT sugli N campioni della prima finestra della traccia audio
dft_c = abs(tes.dft_python_parallela(audio[0:N])) # qui non chiamiamo abs perché viene già calcolato da C

# calcoliamo il limite di frequenza per avere la banda al 99% del segnale
max_freq_banda = tes.get_limite_banda(tes.get_spettro(dft_c), Df, 99)
print(f"La frequenza corrispondente alla banda al 99% è {max_freq_banda}Hz")

# plottiamo lo spettro del segnale
plt.plot(x_axis, tes.shift(dft_c))
plt.grid(True)
plt.axis([-max_freq_banda*1.25, max_freq_banda*1.25, 0, 1.25 * np.max(dft_c)])
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")
plt.show()

#### Confronto delle prestazioni
Dagli esperimenti con le varie implementazione della DFT effettuati abbiamo constatato che:
- La **FFT** implementata dalla libreria Numpy è, come ci si aspettava, estremamente veloce, nell'ordine dei pochi millesimi di secondo per processare una finestra da 20.000 campioni
- La **DFT** è estremamente più lenta, la sua implementazione in Python per eleborare la stessa finestra richiede diverse decine di minuti.
- Anche un'implementazione della **DFT** scritta in un linguaggio decisamente più performante (il **linguaggio C**) richiede comunque diverse decine di secondi per elaborare la trasformata della stessa finestra
- Persino un'implementazione della DFT che fa uso di un tipo di hardware estremamente più performante rispetto a un semplice processore (la **GPU**), non è in grado di avvicinarsi alle prestazioni della FFT. Con quest'ultimo metodo infatti arriviamo a calcolare la DFT in alcune centinaia di millesimi di secondo

Risulta chiaro che la DFT calcolata con il suo algoritmo più semplice risulta non utilizzabile in qualsiasi applicazione reale

### 3) Spettro di energia

Adesso andremo ad analizzare 2 tracce audio (un fremmento della colonna sonora di **Interstellar** e uno di **Bohemian Rhapsody**) attraverso il loro **spettro di energia**, per analizzare la differte distrubizione di energia nelle varie bande di frequenza di due brani appartenenti a generi musicali molto diversi.

Nelle seguenti implementazioni plotteremo i grafici dello spettro di energia per ogni finestra da 0.5 secondi di ogni traccia audio, usando la scala lineare e quella logaritmica.

#### Preparazione dei file audio

In [ ]:
import time
from IPython.display import clear_output
#operazioni preliminari
file_audio1 = str(os.path.join('input', 'CornfieldChase.oga'))
file_audio2 = str(os.path.join('input', 'QueenBohemianRhapsody.oga'))

audio1, fs1 = soundfile.read(file_audio1)
audio1 = np.array(audio1[:, 0], dtype=np.float32)
audio2, fs2 = soundfile.read(file_audio2)
audio2 = np.array(audio2[:, 0], dtype=np.float32)

len_finestra = 0.5
N1 = int(fs1*len_finestra)
N2 = int(fs2*len_finestra)

Df1 = 1/(N1*1/fs1)
Df2 = 1/(N2*1/fs2)
x_axis1 = [x*Df1 for x in range(int(-N1/2),int(N1/2))]
x_axis2 = [x*Df2 for x in range(int(-N2/2),int(N2/2))]

#### Calcolo spettro di energia su tutte le finestre

##### Prima traccia

In [ ]:
display(Audio(file_audio1))

In [ ]:
max_freq_banda1 = 0

for i in range(0,int(len(audio1)/N1)):
    clear_output(wait=True)

    energy_spectrum1 = np.power(np.abs(np.fft.fft(audio1[i*N1:i*N1 + N1])), 2)
    max_freq_banda1 = max(tes.get_limite_banda(energy_spectrum1, Df1, 99), max_freq_banda1)

    plt.plot(x_axis1, tes.shift(energy_spectrum1))
    plt.grid(True)
    plt.axis([-max_freq_banda1*1.25, max_freq_banda1*1.25, 0, 1.25 * np.max(energy_spectrum1)])
    plt.title(f"Cornfield Chase - Finestra: {len_finestra} s - Inizio: {i*N1/fs1} s")
    plt.xlabel("Frequency [Hz]")
    plt.ylabel("Energy")
    plt.show()
    time.sleep(len_finestra)

In [ ]:
max_freq_banda1 = 0

for i in range(0,int(len(audio1)/N1)):
    clear_output(wait=True)

    energy_spectrum1 = np.square(np.abs(np.fft.fft(audio1[i*N1:i*N1 + N1])))
    max_freq_banda1 = max(tes.get_limite_banda(energy_spectrum1, Df1, 99), max_freq_banda1)

    plt.semilogy(x_axis1, tes.shift(energy_spectrum1))
    plt.grid(True)
    plt.ylim(0.75*np.min(energy_spectrum1), 1.25 * np.max(energy_spectrum1))
    plt.title(f"Cornfield Chase - Finestra: {len_finestra} s - Inizio: {i*N1/fs1} s")
    plt.xlabel("Frequency [Hz]")
    plt.ylabel("Energy (Log)")
    plt.show()
    time.sleep(len_finestra)

#### Seconda traccia

In [ ]:
display(Audio(file_audio2))

In [ ]:
for i in range(0,int(len(audio2)/N2)):
    clear_output(wait=True)

    energy_spectrum2 = np.power(np.abs(np.fft.fft(audio2[i*N2:i*N2 + N2])), 2)

    plt.plot(x_axis2, tes.shift(energy_spectrum2))
    plt.grid(True)
    plt.axis([-1.25*tes.get_limite_banda(energy_spectrum2, Df2, 95), tes.get_limite_banda(energy_spectrum2, Df2, 95)*1.25, 0, 1.25 * np.max(energy_spectrum2)])
    plt.xlabel("Frequency [Hz]")
    plt.ylabel("Energy")
    plt.title(f"Bohemian Rhapsody - Finestra: {len_finestra} s - Inizio: {i*N2/fs2} s")
    plt.show()
    time.sleep(len_finestra)

In [ ]:
for i in range(0,int(len(audio2)/N2)):
    clear_output(wait=True)

    energy_spectrum2 = np.square(np.abs(np.fft.fft(audio1[i*N2:i*N2 + N2])))

    plt.semilogy(x_axis2, tes.shift(energy_spectrum2))
    plt.grid(True)
    plt.ylim(0.75*np.min(energy_spectrum2), 1.25 * np.max(energy_spectrum2))
    plt.title(f"Bohemian Rhapsody - Finestra: {len_finestra} s - Inizio: {i*N2/fs2} s")
    plt.xlabel("Frequency [Hz]")
    plt.ylabel("Energy (Log)")
    plt.show()
    time.sleep(len_finestra)